# Electron tomography workflow
**Microscopy and Spectroscopy in Nanosystems, UA, 2026**\
Robin Girod, EMAT, robin.girod@uantwerpen.be


Run the cells from top to bottom; some sections are alternatives or optional, most cells can be rerun safely.

This notebook is intended to be run on a Google Colab GPU runtime, other workflows are available on the [EMAT Tomo](https://github.com/EMAT-tomography/EMAT-tomo2026-python) github.

We will first install the required dependencies to the Colab environment:

In [1]:
import sys, os, subprocess
from google.colab import output, userdata, drive

# Colab-specific library install
!pip install -q astra-toolbox stackview ipycanvas==0.11.0 mrcfile rosettasciio
!git clone --depth 1 https://github.com/EMAT-tomography/EMAT-tomo2026-python.git
TOOLBOX = "/content/EMAT-tomo2026-python"
if TOOLBOX not in sys.path:
    sys.path.insert(0, TOOLBOX)

output.enable_custom_widget_manager()          # needed for stackview/ipycanvas widgets

import numpy as np
import matplotlib.pyplot as plt
import rgtomo as rgt
%matplotlib inline

print("rgtomo", rgt.__version__, "| ASTRA CUDA available:", rgt.has_cuda())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.8/255.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.5/326.5 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.0/519.0 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 128.4 MB/s eta 0:00:00
Cloning into 'EMAT-tomo2026-python'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 23 (delta 0), reused 18 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 422.04 KiB | 1.46 MiB/s, done.
rgtomo 0.1.0 | ASTRA CUDA available: True


Then, download the sample data and unzip them in `/content/demo_data/`:

In [45]:
!gdown 'https://drive.google.com/file/d/1Kd4HBUxN0Trzmytsf5RCDxxHkz8JY41Q/view?usp=sharing' --fuzzy -O /content/demo_data.zip
!mkdir /content/demo_data
!unzip /content/demo_data.zip -d /content/demo_data
!rm /content/demo_data.zip

Downloading...
From (original): https://drive.google.com/uc?id=1Kd4HBUxN0Trzmytsf5RCDxxHkz8JY41Q
From (redirected): https://drive.google.com/uc?id=1Kd4HBUxN0Trzmytsf5RCDxxHkz8JY41Q&confirm=t&uuid=520e2635-f786-4dea-9d09-48849467b77c
To: /content/demo_data.zip
100% 81.5M/81.5M [00:02<00:00, 29.8MB/s]
Archive:  /content/demo_data.zip
  inflating: /content/demo_data/ChiralNR_tiltAngles.csv  
  inflating: /content/demo_data/ChiralNR_tiltSeries.tiff  


---
## Loading data and preprocessing
In the toolbox, the `uniloader` can handle files from folders and movies in `.emi`, `.tif`, or `.emd` format.

For the practical, we will only load `.tif` series with companion `.csv` holding the tilt angles.\
In this example the pixel size is set by `series_raw.pixel_size = 0.2744`, but it would be read from the files in the other formats.

In [47]:
fname = "/content/demo_data/ChiralNR_tiltSeries.tiff"
fangles = "/content/demo_data/ChiralNR_tiltAngles.csv" # angles file, optional, None if unused
ftype = 'tif'      # 'fast', 'manual', 'tif'

series_raw = rgt.uniloader(fname, fangles, ftype)
series_raw.pixel_size = 0.2744 # in nm

print(series_raw)
rgt.slice_viewer(series_raw, zoom_factor = 1)

Loaded 52 projections with 1.000 nm pixel size
TiltSeries(52 projections of 1024 x 1024 px, pixel size 0.2744 nm, angles -77.0..75.9 deg, dtype float32)


### Preprocessing and initial alignment with cross-correlation
**Binning.** Think about your required resolution, a rule of thumb is to have at least 4-5 pixels for the
smallest features to be analyzed. It is not advised to keep more than this for size and noise considerations.

**Alignment.** For isolated particles, it works best to select one angle to which all projections will be
aligned. For slab-like cases, alignment to the previous image typically works better.

**Denoising.** A simple gaussian is often sufficient here.

**YOUR TASK:** With the example NR, the smallest features are ~ 2 nm. Play with the `rescale_factor`, `slice_to_align_to`, `edge`, `sigma_gauss_filter`, and `iterations` parameters until the series appears well aligned and has an adequate pixel size.

In [50]:
# Parameters
rescale_factor = 0.5          # (float): Factor of the new pixel size
slice_to_align_to = 17    # (int): Projection index (0-based) or None to align to neighbor
edge = 0                    # (int): gaussian sigma after sobel filtering. if 0: no edge filter
iterations = 2              # (int): Iterations for x-correlation, typically 1-3
sigma_gauss_filter = 1      # (float): Denoising with gaussian filter, 0 for none, otherwise std of the gaussian kernel

# Run code
series = series_raw.copy()

# normalize each projection -> not always a correct assumption!
series.data = rgt.normalize_intensities(series.data)

if rescale_factor < 1:
    series = rgt.imresize_stack(series, rescale_factor)
    print(f"new pixel size: {series.pixel_size:.3f} nm")

series.data = rgt.gaussfilter_slice(series.data, sigma_gauss_filter)

stack = None
if iterations > 0:
    series.data, _, _, stack = rgt.align_stack_xcorr(series, slice_to_align_to, 0, edge,
                                                     iterations)
series.data = rgt.rescale(series.data)

# Check alignment: aligned series (left) and data used for alignment (right)
rgt.slice_viewer(series.data if stack is None else np.concatenate([series.data, rgt.rescale(stack)], axis=2))

new pixel size: 0.549 nm
xcorr alignment, iteration 1
Aligning without windowing...
Aligning with regular images...
Finding shifts via cross-correlation...


xcorr:   0%|          | 0/52 [00:00<?, ?it/s]

xcorr alignment, iteration 2
Aligning without windowing...
Aligning with regular images...
Finding shifts via cross-correlation...


xcorr:   0%|          | 0/52 [00:00<?, ?it/s]

## Crop and rotate
The tilt series must be rotated to be vertical. Here we only correct for the fact that the scan direction has
typically a 90° difference between the Osiris and the Titans.
**/!\ rotate by 1x90° only for the Titans, 0 for the Osiris.**

**YOUR TASK:** Draw the ROI on the projection (or use the sliders), then run the next cell to crop.

In [51]:
rotate = 1   # rotation of n times 90°

series_test = series.copy()
series_test.data = rgt.rot90_stack(series_test.data, rotate)
crop_gui = rgt.crop_stack_gui(series_test)
crop_gui

In [52]:
series_test = crop_gui.result()
print("ROI (x0, y0, width, height):", crop_gui.rect, "->", series_test)

ROI (x0, y0, width, height): (77, 129, 324, 342) -> TiltSeries(52 projections of 342 x 324 px, pixel size 0.5488 nm, angles -77.0..75.9 deg, dtype float32)


---
## Final processing
### Final background removal and second alignment
**Background removal** (optional): the script will fit a background function, subtract it, and remove parts
that are below an intensity threshold. Decrease `BGSUBTRACT_THRESHOLD` to remove less.

**YOUR TASK:** Test different `BGSUBTRACT_THRESHOLD` values so that the background is removed but does not crop out intensities from the particles. Also confirm the alignment again, with the same parameters as above.

In [60]:
'''Background params'''
remove_background = True        # (bool)
BGSUBTRACT_THRESHOLD = 0.8      # factor of otsu graythresh to keep, higher removes more, default 0.5

'''Alignment params'''
slice_to_align_to = 15      # (int): Projection index (0-based) or None to align to neighbor
edge = 0                    # (int): gaussian sigma after sobel filtering. if 0: no edge filter
iterations = 3              # (int): Iterations for x-correlation, typically 1-3

'''RUN'''
series_final = series_test.copy()

if remove_background:
    series_final.data = rgt.background_subtract(series_final.data, BGSUBTRACT_THRESHOLD)
    series_final.data = rgt.normalize_intensities(series_final.data)   # norm each proj to mean 1 std 1
series_final.data = rgt.rescale(series_final.data)
if iterations > 0:
    series_final.data, _, _, stack = rgt.align_stack_xcorr(series_final, slice_to_align_to, 0, edge,
                                                           iterations)

rgt.slice_viewer(series_final.data if stack is None else np.concatenate([series_final.data, rgt.rescale(stack)], axis=2))

Removing background:   0%|          | 0/52 [00:00<?, ?it/s]

xcorr alignment, iteration 1
Aligning without windowing...
Aligning with regular images...
Finding shifts via cross-correlation...


xcorr:   0%|          | 0/52 [00:00<?, ?it/s]

xcorr alignment, iteration 2
Aligning without windowing...
Aligning with regular images...
Finding shifts via cross-correlation...


xcorr:   0%|          | 0/52 [00:00<?, ?it/s]

xcorr alignment, iteration 3
Aligning without windowing...
Aligning with regular images...
Finding shifts via cross-correlation...


xcorr:   0%|          | 0/52 [00:00<?, ?it/s]

## Align tilt axis
Three slices are reconstructed: shift/rotate the tilt axis (red line) and press *Preview* until the arcs of a
misaligned axis disappear. Run the next cell to apply. (This tool can be used at any stage, e.g. again after the
self-consistency refinements.)

**YOUR TASK:** Play with the contrast until arc artefacts are clear, then minimize them with the translation and rotation slides.

In [61]:
tilt_gui = rgt.align_tilt_axis_gui(series_final)
tilt_gui

In [62]:
series_final = tilt_gui.result()
print(f"tilt axis translation {tilt_gui.translation} px, rotation {tilt_gui.rotation} deg")

tilt axis translation 2.0 px, rotation 10.3 deg


## Cupping correction (optional, for large particles)
Increase the correction strength until the reconstructed slice (EM/SIRT) has no darker core.

In [64]:
cupping_gui = rgt.cupping_correction_gui(series_final)
cupping_gui

In [65]:
series_final = cupping_gui.result()
series_final.data = rgt.remove_bkg_median(series_final.data)

## Reconstruction

In [71]:
alg = 'fbp'   # 'em', 'sirt', 'fbp'
it = 1        # iterations, 25 for EM, up to 50-100 for SIRT

series_rec = series_final                          # tilt series to reconstruct
result = rgt.reconstruct(series_rec, alg, it, True, 30)  # apodization, 30 px padding
rgt.orthoslice_viewer(result, zoom_factor=1)

Performing reconstruction using FBP algorithm...


FBP reconstruction:   0%|          | 0/342 [00:00<?, ?it/s]

## Post-processing
### Align and post process
Simple filtering in 3D + alignment of the major axis of reconstruction using the first principal component.
If the reconstruction is not a rod, leave the PCA line commented.

In [67]:
gaussian_sigma = 1            # std of the gaussian kernel, in vx
median_kernel = (1, 1, 1)     # size of the median kernel

result_processed = result.astype(np.float32)
result_processed = rgt.gaussian_filter_3d(result_processed, gaussian_sigma)
result_processed = rgt.median_filter_3d(result_processed, median_kernel)
result_processed = rgt.align_z_pca(result_processed, (1, 99))  # lower and upper percentile thresholds for outliers

rgt.orthoslice_viewer(result_processed)